# Comprehensive Prompt Format Study

- Testing prompt format robustness across quantization methods
- 8 configs x 5 formats x 30 questions = 1200 generations

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch',
    'transformers==4.51.3',
    'accelerate',
    'bitsandbytes',
    'autoawq',
    'exllamav2',
    'huggingface-hub',
    'datasets',
    'rapidfuzz'
], check=True)

print("Dependencies installed")

In [ ]:
import gc
import json
import time
import re
import torch
import random
from pathlib import Path
from typing import Dict, List, Tuple
from collections import defaultdict
from datasets import load_dataset
from rapidfuzz import fuzz

print("Imports complete")

In [ ]:
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
RANDOM_SEED = 42
OUTPUT_DIR = Path('/kaggle/working/comprehensive_study')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print(f"Device: {DEVICE}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()

def get_vram_usage():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        return allocated, total
    return 0, 0

def save_results(data: Dict, filename: str):
    output_path = OUTPUT_DIR / filename
    with open(output_path, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"Saved: {filename}")

print("Utility functions defined")

In [ ]:
def load_fewshot_examples():
    print("Loading few-shot examples from training splits...")
    
    fewshot_examples = {}
    
    try:
        squad_train = load_dataset('squad_v2', split='train', streaming=False)
        squad_train_list = list(squad_train.select(range(500)))
        
        squad_train_answerable = [s for s in squad_train_list if len(s['answers']['text']) > 0]
        squad_train_short = [s for s in squad_train_answerable if 1 <= len(s['answers']['text'][0].split()) <= 3]
        
        fewshot_examples['simple_factual_1'] = squad_train_short[10]
        fewshot_examples['simple_factual_2'] = squad_train_short[25]
        
        squad_train_medium = [s for s in squad_train_answerable if 3 <= len(s['answers']['text'][0].split()) <= 8]
        fewshot_examples['context_extraction'] = squad_train_medium[15]
        
        print(f"Loaded {len(fewshot_examples)} SQuAD train examples")
        
    except Exception as e:
        print(f"Error loading SQuAD train: {e}")
        fewshot_examples['simple_factual_1'] = {
            'question': 'What is the capital of France?',
            'answers': {'text': ['Paris']},
            'context': 'Paris is the capital and most populous city of France.'
        }
        fewshot_examples['simple_factual_2'] = {
            'question': 'How many continents are there?',
            'answers': {'text': ['7']},
            'context': 'There are 7 continents on Earth.'
        }
        fewshot_examples['context_extraction'] = {
            'question': 'Who invented the telephone?',
            'answers': {'text': ['Alexander Graham Bell']},
            'context': 'Alexander Graham Bell invented the telephone in 1876.'
        }
    
    try:
        drop_train = load_dataset('drop', split='train', streaming=False)
        drop_train_list = list(drop_train.select(range(200)))
        drop_train_numerical = [s for s in drop_train_list if 'number' in s['answers_spans']['types']]
        
        fewshot_examples['numerical'] = drop_train_numerical[8]
        print("Loaded DROP train example")
        
    except Exception as e:
        print(f"Error loading DROP train: {e}")
        fewshot_examples['numerical'] = {
            'question': 'What is 20% of 150?',
            'answers_spans': {'spans': ['30']},
            'passage': 'To calculate 20% of 150, multiply 150 by 0.20 to get 30.'
        }
    
    return fewshot_examples

FEWSHOT_EXAMPLES = load_fewshot_examples()

In [ ]:
def load_validation_questions():
    print("Loading validation questions...")
    all_questions = []
    
    try:
        print("Loading SQuAD v2 validation...")
        squad = load_dataset('squad_v2', split='validation', streaming=False)
        squad_list = list(squad.select(range(min(500, len(squad)))))
        
        squad_answerable = [s for s in squad_list if len(s['answers']['text']) > 0]
        squad_factual = [s for s in squad_answerable if len(s['answers']['text'][0].split()) <= 3]
        
        random.seed(RANDOM_SEED)
        sampled_factual = random.sample(squad_factual, min(5, len(squad_factual)))
        
        for idx, sample in enumerate(sampled_factual):
            all_questions.append({
                'id': f'simple_factual_{idx+1}',
                'category': 'simple_factual',
                'question': sample['question'],
                'answer': sample['answers']['text'][0],
                'context': sample['context'],
                'source': 'squad_v2'
            })
        
        print(f"Loaded {len(sampled_factual)} simple factual questions")
        
    except Exception as e:
        print(f"Error loading simple factual: {e}")
    
    try:
        print("Loading HotpotQA validation...")
        hotpot = load_dataset('hotpot_qa', 'distractor', split='validation', streaming=False)
        hotpot_list = list(hotpot.select(range(min(100, len(hotpot)))))
        
        random.seed(RANDOM_SEED)
        sampled_hotpot = random.sample(hotpot_list, min(5, len(hotpot_list)))
        
        for idx, sample in enumerate(sampled_hotpot):
            context_str = ' '.join([' '.join(sentences) for sentences in sample['context']['sentences']])
            all_questions.append({
                'id': f'multi_hop_{idx+1}',
                'category': 'multi_hop',
                'question': sample['question'],
                'answer': sample['answer'],
                'context': context_str,
                'source': 'hotpot_qa'
            })
        
        print(f"Loaded {len(sampled_hotpot)} multi-hop questions")
        
    except Exception as e:
        print(f"Error loading multi-hop: {e}")
    
    try:
        print("Loading DROP validation...")
        drop = load_dataset('drop', split='validation', streaming=False)
        drop_list = list(drop.select(range(min(200, len(drop)))))
        
        drop_numerical = [s for s in drop_list if 'number' in s['answers_spans']['types']]
        
        random.seed(RANDOM_SEED)
        sampled_drop = random.sample(drop_numerical, min(5, len(drop_numerical)))
        
        for idx, sample in enumerate(sampled_drop):
            all_questions.append({
                'id': f'numerical_{idx+1}',
                'category': 'numerical',
                'question': sample['question'],
                'answer': sample['answers_spans']['spans'][0],
                'context': sample['passage'],
                'source': 'drop'
            })
        
        print(f"Loaded {len(sampled_drop)} numerical questions")
        
    except Exception as e:
        print(f"Error loading numerical: {e}")
    
    try:
        print("Loading NarrativeQA validation...")
        narrative = load_dataset('narrativeqa', split='validation', streaming=False)
        narrative_list = list(narrative.select(range(min(50, len(narrative)))))
        
        random.seed(RANDOM_SEED)
        sampled_narrative = random.sample(narrative_list, min(5, len(narrative_list)))
        
        for idx, sample in enumerate(sampled_narrative):
            all_questions.append({
                'id': f'context_dependent_{idx+1}',
                'category': 'context_dependent',
                'question': sample['question']['text'],
                'answer': sample['answers'][0]['text'],
                'context': sample['document']['summary']['text'],
                'source': 'narrativeqa'
            })
        
        print(f"Loaded {len(sampled_narrative)} context-dependent questions")
        
    except Exception as e:
        print(f"Error loading context-dependent: {e}")
    
    try:
        print("Loading unanswerable questions...")
        squad_unanswerable = [s for s in squad_list if len(s['answers']['text']) == 0]
        sampled_unanswerable = random.sample(squad_unanswerable, min(5, len(squad_unanswerable)))
        
        for idx, sample in enumerate(sampled_unanswerable):
            all_questions.append({
                'id': f'unanswerable_{idx+1}',
                'category': 'unanswerable',
                'question': sample['question'],
                'answer': 'unanswerable',
                'context': sample['context'],
                'source': 'squad_v2'
            })
        
        print(f"Loaded {len(sampled_unanswerable)} unanswerable questions")
        
    except Exception as e:
        print(f"Error loading unanswerable: {e}")
    
    try:
        print("Loading ELI5 validation...")
        eli5 = load_dataset('eli5', split='validation_eli5', streaming=False)
        eli5_list = list(eli5.select(range(min(100, len(eli5)))))
        
        eli5_filtered = [s for s in eli5_list if len(s['answers']['text']) > 0]
        
        random.seed(RANDOM_SEED)
        sampled_eli5 = random.sample(eli5_filtered, min(5, len(eli5_filtered)))
        
        for idx, sample in enumerate(sampled_eli5):
            all_questions.append({
                'id': f'longform_{idx+1}',
                'category': 'longform',
                'question': sample['title'],
                'answer': sample['answers']['text'][0][:200],
                'context': '',
                'source': 'eli5'
            })
        
        print(f"Loaded {len(sampled_eli5)} long-form questions")
        
    except Exception as e:
        print(f"Error loading long-form: {e}")
    
    print(f"Total loaded: {len(all_questions)} validation questions")
    category_counts = defaultdict(int)
    for q in all_questions:
        category_counts[q['category']] += 1
    print("Breakdown by category:")
    for cat, count in sorted(category_counts.items()):
        print(f"  {cat}: {count}")
    
    return all_questions

VALIDATION_QUESTIONS = load_validation_questions()

In [ ]:
def normalize_answer(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    text = ' '.join(text.split())
    return text

def fuzzy_contains_answer(response: str, expected: str) -> Tuple[bool, float]:
    response_norm = normalize_answer(response)
    expected_norm = normalize_answer(expected)
    
    if expected_norm in response_norm:
        return True, 1.0
    
    similarity = fuzz.partial_ratio(expected_norm, response_norm) / 100.0
    
    expected_tokens = set(expected_norm.split())
    response_tokens = set(response_norm.split())
    if expected_tokens and len(expected_tokens & response_tokens) == len(expected_tokens):
        return True, 0.95
    
    return similarity >= 0.85, similarity

def evaluate_response(response: str, expected_answer: str, category: str) -> Dict:
    response_clean = response.strip().lower()
    words = response_clean.split()
    
    if expected_answer == 'unanswerable':
        refusal_phrases = ['cannot', 'not mentioned', 'does not', 'no information', 'unanswerable', 'not provided', 'not specified']
        contains_answer = any(phrase in response_clean for phrase in refusal_phrases)
        similarity_score = 1.0 if contains_answer else 0.0
    else:
        contains_answer, similarity_score = fuzzy_contains_answer(response, expected_answer)
    
    has_token_loop = False
    if len(words) > 5:
        for i in range(len(words) - 2):
            if words[i] == words[i+1] == words[i+2]:
                has_token_loop = True
                break
    
    special_char_ratio = sum(1 for c in response_clean[:100] 
                            if not c.isalnum() and c not in ' \n.,!?-:;\'\"') / max(len(response_clean[:100]), 1)
    has_garbage = special_char_ratio > 0.4
    
    if category == 'longform':
        reasonable_length = len(words) >= 10
    else:
        reasonable_length = 1 <= len(words) <= 150
    
    if len(words) > 5:
        unique_ratio = len(set(words)) / len(words)
    else:
        unique_ratio = 1.0
    
    very_low_diversity = unique_ratio < 0.3
    
    pass_fail = (
        contains_answer and
        not has_token_loop and
        not has_garbage and
        reasonable_length and
        not very_low_diversity
    )
    
    return {
        'pass_fail': pass_fail,
        'contains_answer': contains_answer,
        'answer_similarity': similarity_score,
        'word_diversity': unique_ratio,
        'very_low_diversity': very_low_diversity,
        'has_garbage': has_garbage,
        'has_token_loop': has_token_loop,
        'reasonable_length': reasonable_length,
        'length_words': len(words),
        'response_preview': response_clean[:150]
    }

print("Evaluation functions defined")

In [ ]:
class PromptFormats:
    
    @staticmethod
    def instruct_minimal(question: str, context: str = None) -> str:
        if context:
            return f"[INST] {context}\n\n{question} [/INST]"
        return f"[INST] {question} [/INST]"
    
    @staticmethod
    def instruct_verbose(question: str, context: str = None) -> str:
        if context:
            return f"""[INST] Answer the following question using only the information provided. Be direct and concise.

Context: {context}

Question: {question}

Answer: [/INST]"""
        return f"""[INST] Answer the following question concisely based on your knowledge.

Question: {question}

Answer: [/INST]"""
    
    @staticmethod
    def instruct_chat(question: str, context: str = None) -> str:
        if context:
            return f"[INST] Context: {context}\n\nBased on this, {question} [/INST]"
        return f"[INST] {question} [/INST]"
    
    @staticmethod
    def instruct_system(question: str, context: str = None) -> str:
        if context:
            return f"""[INST] <<SYS>>
You are a helpful assistant. Answer questions based on the given context.
<</SYS>>

Context: {context}

Question: {question} [/INST]"""
        return f"""[INST] <<SYS>>
You are a helpful assistant.
<</SYS>>

{question} [/INST]"""
    
    @staticmethod
    def instruct_qa(question: str, context: str = None) -> str:
        if context:
            return f"[INST] Context: {context}\n\nQ: {question}\nA: [/INST]"
        return f"[INST] Q: {question}\nA: [/INST]"
    
    @staticmethod
    def base_minimal(question: str, context: str = None) -> str:
        if context:
            return f"{context}\n\nQ: {question}\nA:"
        return f"Q: {question}\nA:"
    
    @staticmethod
    def base_fewshot_short(question: str, context: str = None) -> str:
        ex1 = FEWSHOT_EXAMPLES['simple_factual_1']
        
        if context:
            return f"""Context: {ex1['context']}

Q: {ex1['question']}
A: {ex1['answers']['text'][0]}

Context: {context}

Q: {question}
A:"""
        return f"""Q: {ex1['question']}
A: {ex1['answers']['text'][0]}

Q: {question}
A:"""
    
    @staticmethod
    def base_fewshot_long(question: str, context: str = None) -> str:
        ex1 = FEWSHOT_EXAMPLES['simple_factual_1']
        ex2 = FEWSHOT_EXAMPLES['simple_factual_2']
        
        if context:
            return f"""Context: {ex1['context']}

Q: {ex1['question']}
A: {ex1['answers']['text'][0]}

Context: {ex2['context']}

Q: {ex2['question']}
A: {ex2['answers']['text'][0]}

Context: {context}

Q: {question}
A:"""
        return f"""Q: {ex1['question']}
A: {ex1['answers']['text'][0]}

Q: {ex2['question']}
A: {ex2['answers']['text'][0]}

Q: {question}
A:"""
    
    @staticmethod
    def base_labeled(question: str, context: str = None) -> str:
        if context:
            return f"Context: {context}\n\nQuestion: {question}\nAnswer:"
        return f"Question: {question}\nAnswer:"
    
    @staticmethod
    def base_completion(question: str, context: str = None) -> str:
        if context:
            return f"{context}\n\nThe answer to \"{question}\" is:"
        return f"The answer to \"{question}\" is:"

INSTRUCT_FORMATS = {
    'minimal': PromptFormats.instruct_minimal,
    'verbose': PromptFormats.instruct_verbose,
    'chat': PromptFormats.instruct_chat,
    'system': PromptFormats.instruct_system,
    'qa': PromptFormats.instruct_qa
}

BASE_FORMATS = {
    'minimal': PromptFormats.base_minimal,
    'fewshot_short': PromptFormats.base_fewshot_short,
    'fewshot_long': PromptFormats.base_fewshot_long,
    'labeled': PromptFormats.base_labeled,
    'completion': PromptFormats.base_completion
}

print("Prompt formats defined")

In [ ]:
def load_transformers_model(model_name: str, quantization: str):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    if quantization == 'fp16':
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map='auto',
            use_cache=True
        )
    elif quantization == 'nf4':
        from transformers import BitsAndBytesConfig
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map='auto',
            use_cache=True
        )
    else:
        raise ValueError(f"Unsupported quantization: {quantization}")
    
    model.eval()
    return model, tokenizer

def load_awq_model(model_name: str):
    from awq import AutoAWQForCausalLM
    from transformers import AutoTokenizer
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoAWQForCausalLM.from_quantized(
        model_name,
        fuse_layers=True,
        use_cache=True
    )
    model.eval()
    return model, tokenizer

def load_gptq_model(model_name: str):
    from exllamav2 import ExLlamaV2, ExLlamaV2Config, ExLlamaV2Cache, ExLlamaV2Tokenizer
    from huggingface_hub import snapshot_download
    
    model_dir = snapshot_download(
        model_name,
        allow_patterns=["*.json", "*.safetensors", "*.model"]
    )
    
    config = ExLlamaV2Config()
    config.model_dir = model_dir
    config.prepare()
    config.max_seq_len = 4096
    
    model = ExLlamaV2(config)
    cache = ExLlamaV2Cache(model, lazy=True, max_seq_len=2048)
    model.load_autosplit(cache)
    
    tokenizer = ExLlamaV2Tokenizer(config)
    
    return model, tokenizer, cache

def get_model_device(model):
    if hasattr(model, 'device'):
        return model.device
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print("Model loading functions defined")

In [ ]:
def generate_transformers(model, tokenizer, prompt: str, temperature: float, rep_penalty: float) -> Tuple[str, float]:
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)
    
    device = get_model_device(model)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=4096)
    input_length = inputs['input_ids'].shape[1]
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    start_time = time.perf_counter()
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=True,
            temperature=temperature,
            top_k=50,
            top_p=0.9,
            repetition_penalty=rep_penalty,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )
    
    generation_time = (time.perf_counter() - start_time) * 1000
    
    generated_tokens = outputs[0][input_length:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    return response.strip(), generation_time

def generate_exllama(model, tokenizer, cache, prompt: str, temperature: float, rep_penalty: float) -> Tuple[str, float]:
    from exllamav2.generator import ExLlamaV2StreamingGenerator, ExLlamaV2Sampler
    
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)
    
    generator = ExLlamaV2StreamingGenerator(model, cache, tokenizer)
    
    settings = ExLlamaV2Sampler.Settings()
    settings.temperature = temperature
    settings.top_k = 50
    settings.top_p = 0.9
    settings.token_repetition_penalty = rep_penalty
    
    input_ids = tokenizer.encode(prompt)
    
    start_time = time.perf_counter()
    
    generator.begin_stream(input_ids, settings)
    
    output_tokens = []
    for _ in range(128):
        chunk, eos, _ = generator.stream()
        output_tokens.append(chunk)
        if eos:
            break
    
    generation_time = (time.perf_counter() - start_time) * 1000
    
    output = ''.join(output_tokens).strip()
    
    return output, generation_time

print("Generation functions defined")

In [ ]:
def test_model_config(
    model_name: str,
    quantization: str,
    variant: str,
    temperature: float,
    rep_penalty: float
) -> Dict:
    
    config_name = f"{quantization}_{variant}"
    print(f"Testing: {config_name}")
    print(f"Model: {model_name}")
    
    start_time = time.time()
    
    try:
        print("Loading model...")
        load_start = time.time()
        
        cache = None
        if quantization == 'gptq':
            model, tokenizer, cache = load_gptq_model(model_name)
        elif quantization == 'awq':
            model, tokenizer = load_awq_model(model_name)
        else:
            model, tokenizer = load_transformers_model(model_name, quantization)
        
        load_time = time.time() - load_start
        print(f"Model loaded in {load_time:.1f}s")
        
        vram_alloc, vram_total = get_vram_usage()
        print(f"VRAM: {vram_alloc:.2f}GB / {vram_total:.2f}GB")
        
        if variant == 'instruct':
            format_dict = INSTRUCT_FORMATS
        else:
            format_dict = BASE_FORMATS
        
        all_results = []
        format_summaries = {}
        
        for format_name, format_func in format_dict.items():
            print(f"  Format: {format_name}")
            format_results = []
            
            for q in VALIDATION_QUESTIONS:
                try:
                    prompt = format_func(q['question'], q['context'])
                    
                    if quantization == 'gptq':
                        response, gen_time = generate_exllama(
                            model, tokenizer, cache, prompt,
                            temperature, rep_penalty
                        )
                    else:
                        response, gen_time = generate_transformers(
                            model, tokenizer, prompt,
                            temperature, rep_penalty
                        )
                    
                    eval_result = evaluate_response(response, q['answer'], q['category'])
                    
                    result = {
                        'question_id': q['id'],
                        'category': q['category'],
                        'format': format_name,
                        'question': q['question'],
                        'expected_answer': q['answer'],
                        'response': response,
                        'generation_time_ms': gen_time,
                        'source_dataset': q['source'],
                        **eval_result
                    }
                    
                    format_results.append(result)
                    all_results.append(result)
                    
                except Exception as e:
                    print(f"    Error on {q['id']}: {str(e)}")
                    result = {
                        'question_id': q['id'],
                        'category': q['category'],
                        'format': format_name,
                        'error': str(e),
                        'pass_fail': False
                    }
                    format_results.append(result)
                    all_results.append(result)
            
            pass_count = sum(1 for r in format_results if r.get('pass_fail', False))
            format_summaries[format_name] = {
                'pass_rate': pass_count / len(format_results),
                'pass_count': pass_count,
                'total_count': len(format_results),
                'avg_gen_time_ms': sum(r.get('generation_time_ms', 0) for r in format_results) / len(format_results)
            }
            print(f"    Pass rate: {format_summaries[format_name]['pass_rate']:.2%}")
        
        category_summaries = {}
        for category in set(q['category'] for q in VALIDATION_QUESTIONS):
            cat_results = [r for r in all_results if r.get('category') == category]
            pass_count = sum(1 for r in cat_results if r.get('pass_fail', False))
            category_summaries[category] = {
                'pass_rate': pass_count / len(cat_results) if cat_results else 0,
                'pass_count': pass_count,
                'total_count': len(cat_results)
            }
        
        overall_pass = sum(1 for r in all_results if r.get('pass_fail', False))
        
        results = {
            'model_config': {
                'name': model_name,
                'quantization': quantization,
                'variant': variant,
                'temperature': temperature,
                'repetition_penalty': rep_penalty
            },
            'system_info': {
                'vram_allocated_gb': vram_alloc,
                'vram_total_gb': vram_total,
                'load_time_seconds': load_time
            },
            'all_results': all_results,
            'format_summaries': format_summaries,
            'category_summaries': category_summaries,
            'overall_summary': {
                'pass_rate': overall_pass / len(all_results),
                'pass_count': overall_pass,
                'total_count': len(all_results)
            },
            'best_format': max(format_summaries.items(), key=lambda x: x[1]['pass_rate'])[0],
            'worst_format': min(format_summaries.items(), key=lambda x: x[1]['pass_rate'])[0],
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
            'total_test_time_seconds': time.time() - start_time
        }
        
        print(f"Overall pass rate: {results['overall_summary']['pass_rate']:.2%}")
        print(f"Best format: {results['best_format']} ({format_summaries[results['best_format']]['pass_rate']:.2%})")
        print(f"Worst format: {results['worst_format']} ({format_summaries[results['worst_format']]['pass_rate']:.2%})")
        
        print("Cleaning up...")
        del model
        del tokenizer
        if cache:
            del cache
        clear_memory()
        
        print(f"Config complete in {(time.time() - start_time)/60:.1f} minutes")
        
        return results
        
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
        clear_memory()
        
        return {
            'model_config': {
                'name': model_name,
                'quantization': quantization,
                'variant': variant
            },
            'error': str(e),
            'traceback': traceback.format_exc()
        }

print("Main testing pipeline defined")

In [ ]:
print("Starting FP16 Base Model Test")
fp16_base_results = test_model_config(
    model_name='mistralai/Mistral-7B-v0.1',
    quantization='fp16',
    variant='base',
    temperature=0.3,
    rep_penalty=1.15
)
save_results(fp16_base_results, 'fp16_base_results.json')

In [ ]:
print("Starting FP16 Instruct Model Test")
fp16_instruct_results = test_model_config(
    model_name='mistralai/Mistral-7B-Instruct-v0.1',
    quantization='fp16',
    variant='instruct',
    temperature=0.7,
    rep_penalty=1.1
)
save_results(fp16_instruct_results, 'fp16_instruct_results.json')

In [ ]:
print("Starting AWQ Base Model Test")
awq_base_results = test_model_config(
    model_name='TheBloke/Mistral-7B-v0.1-AWQ',
    quantization='awq',
    variant='base',
    temperature=0.3,
    rep_penalty=1.15
)
save_results(awq_base_results, 'awq_base_results.json')

In [ ]:
print("Starting AWQ Instruct Model Test")
awq_instruct_results = test_model_config(
    model_name='TheBloke/Mistral-7B-Instruct-v0.1-AWQ',
    quantization='awq',
    variant='instruct',
    temperature=0.7,
    rep_penalty=1.1
)
save_results(awq_instruct_results, 'awq_instruct_results.json')

In [ ]:
print("Starting NF4 Base Model Test")
nf4_base_results = test_model_config(
    model_name='mistralai/Mistral-7B-v0.1',
    quantization='nf4',
    variant='base',
    temperature=0.3,
    rep_penalty=1.15
)
save_results(nf4_base_results, 'nf4_base_results.json')

In [ ]:
print("Starting NF4 Instruct Model Test")
nf4_instruct_results = test_model_config(
    model_name='mistralai/Mistral-7B-Instruct-v0.1',
    quantization='nf4',
    variant='instruct',
    temperature=0.7,
    rep_penalty=1.1
)
save_results(nf4_instruct_results, 'nf4_instruct_results.json')

In [ ]:
print("Starting GPTQ Base Model Test")
gptq_base_results = test_model_config(
    model_name='TheBloke/Mistral-7B-v0.1-GPTQ',
    quantization='gptq',
    variant='base',
    temperature=0.3,
    rep_penalty=1.15
)
save_results(gptq_base_results, 'gptq_base_results.json')

In [ ]:
print("Starting GPTQ Instruct Model Test")
gptq_instruct_results = test_model_config(
    model_name='TheBloke/Mistral-7B-Instruct-v0.1-GPTQ',
    quantization='gptq',
    variant='instruct',
    temperature=0.7,
    rep_penalty=1.1
)
save_results(gptq_instruct_results, 'gptq_instruct_results.json')

In [ ]:
print("Aggregating all results...")

all_config_results = {
    'fp16_base': fp16_base_results,
    'fp16_instruct': fp16_instruct_results,
    'awq_base': awq_base_results,
    'awq_instruct': awq_instruct_results,
    'nf4_base': nf4_base_results,
    'nf4_instruct': nf4_instruct_results,
    'gptq_base': gptq_base_results,
    'gptq_instruct': gptq_instruct_results
}

aggregated_file = OUTPUT_DIR / "all_results_aggregated.json"
with open(aggregated_file, 'w') as f:
    json.dump(all_config_results, f, indent=2)

print(f"Saved aggregated results to: {aggregated_file}")

In [ ]:
print("COMPREHENSIVE STUDY SUMMARY")
print()

for config_name, data in all_config_results.items():
    if 'error' in data:
        print(f"{config_name}: FAILED - {data['error']}")
        continue
    
    print(f"{config_name}:")
    print(f"  Overall: {data['overall_summary']['pass_rate']:.2%} ({data['overall_summary']['pass_count']}/{data['overall_summary']['total_count']})")
    
    print(f"  Format breakdown:")
    for fmt, stats in data['format_summaries'].items():
        print(f"    {fmt}: {stats['pass_rate']:.2%} ({stats['pass_count']}/{stats['total_count']})")
    
    print(f"  Category breakdown:")
    for cat, stats in sorted(data['category_summaries'].items()):
        print(f"    {cat}: {stats['pass_rate']:.2%}")
    print()

In [ ]:
print("FORMAT COMPARISON ACROSS CONFIGS")
print()

all_format_names = set()
for data in all_config_results.values():
    if 'error' not in data:
        all_format_names.update(data['format_summaries'].keys())

for format_name in sorted(all_format_names):
    print(f"{format_name}:")
    for config_name, data in all_config_results.items():
        if 'error' not in data and format_name in data['format_summaries']:
            rate = data['format_summaries'][format_name]['pass_rate']
            count = data['format_summaries'][format_name]['pass_count']
            total = data['format_summaries'][format_name]['total_count']
            print(f"  {config_name}: {rate:.2%} ({count}/{total})")
    print()

In [ ]:
print("QUANTIZATION METHOD COMPARISON")
print()

quant_methods = ['fp16', 'awq', 'nf4', 'gptq']

for quant in quant_methods:
    base_key = f"{quant}_base"
    instruct_key = f"{quant}_instruct"
    
    print(f"{quant.upper()}:")
    
    if base_key in all_config_results and 'error' not in all_config_results[base_key]:
        rate = all_config_results[base_key]['overall_summary']['pass_rate']
        print(f"  Base: {rate:.2%}")
    
    if instruct_key in all_config_results and 'error' not in all_config_results[instruct_key]:
        rate = all_config_results[instruct_key]['overall_summary']['pass_rate']
        print(f"  Instruct: {rate:.2%}")
    print()

In [ ]:
print("QUESTION CATEGORY DIFFICULTY")
print()

categories = set()
for data in all_config_results.values():
    if 'error' not in data:
        categories.update(data['category_summaries'].keys())

for cat in sorted(categories):
    print(f"{cat}:")
    for config_name, data in all_config_results.items():
        if 'error' not in data and cat in data['category_summaries']:
            rate = data['category_summaries'][cat]['pass_rate']
            print(f"  {config_name}: {rate:.2%}")
    print()

print("Comprehensive study complete.")